# Exploration sim_v2

Connexion pipeline + visualisation des scenarios, du dataset de simulation
et des resultats LOO restitution.


## 1. Setup


In [ ]:
from pathlib import Path
import sys

# Racine release (dossier de ce notebook)
ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline").is_dir():
    # si le kernel a un autre cwd, remonter depuis le fichier
    ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.paths import Paths, release_root
from src.pipeline.connection import PipelineFactory
from src.pipeline.engine import ConnectionPipeline

paths = Paths(ROOT).ensure()
print("ROOT     :", paths.root)
print("DB       :", paths.main_db, "exists=", paths.main_db.exists())
print("pipeline :", paths.pipeline)


ModuleNotFoundError: No module named 'duckdb'

## 2. Connexion


In [ ]:
# Connexion lecture/ecriture sur la base principale + YAML pipeline/
cp = PipelineFactory(paths).open(read_only=False)
print("project_dir :", cp.project_dir)
print("objets pipeline YAML :", len(cp.pipeline))


## 3. Toutes les relations


In [ ]:
import duckdb

def list_relations(cp, like: str | None = None):
    """Liste tables et vues de la base ouverte."""
    q = """
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = current_schema()
        ORDER BY table_type, table_name
    """
    df = cp.con.execute(q).df()
    if like:
        df = df[df["table_name"].str.contains(like, case=False, na=False)]
    return df

rels = list_relations(cp)
display(rels)
print(f"{len(rels)} relations")

display(rels.head(80))


## 4. Objets sim_v2 (scenarios, dataset, restitution, loo)


In [ ]:
keys = ("scenario", "dataset", "restitution", "loo", "rank", "sales", "pivot")
mask = rels["table_name"].str.lower().apply(lambda n: any(k in n for k in keys))
display(rels[mask])


## 5. Catalogue de scenarios `t_scenarios`


In [ ]:
if cp.relation_exists("t_scenarios"):
    sc = cp.table_view("t_scenarios").df()
    display(sc.head(20))
    print("n_scenarios =", len(sc))
    print(sc.dtypes)
else:
    print("t_scenarios absente")


## 6. Resultats de simulation `t_dataset_pivot`


In [ ]:
if cp.relation_exists("t_dataset_pivot"):
    pivot = cp.table_view("t_dataset_pivot").df()
    # colonnes principales
    core = [c for c in [
        "scenario_id", "hotel_code", "solution", "scenario_removed_natures",
        "metres_lineaires", "nombre_guests_par_mois",
        "nombre_ventes_par_mois", "montant_ventes_par_mois",
        "montant_marge_par_mois", "montant_marge_selon_coef_par_mois",
        "nombre_natures",
    ] if c in pivot.columns]
    display(pivot[core].head(30))
    print("shape", pivot.shape)
    print("hotels", pivot["hotel_code"].nunique(), "scenarios", pivot["scenario_id"].nunique())
else:
    print("t_dataset_pivot absente — lancer sim-v2-build")


## 7. Observation (scenario vide) par hotel


In [ ]:
if cp.relation_exists("t_dataset_pivot"):
    obs = cp.con.execute("""
        SELECT hotel_code, solution,
               montant_ventes_par_mois,
               montant_marge_par_mois,
               montant_marge_selon_coef_par_mois,
               metres_lineaires,
               nombre_natures
        FROM t_dataset_pivot
        WHERE COALESCE(LEN(scenario_removed_natures), 0) = 0
        ORDER BY hotel_code
    """).df()
    display(obs)


## 8. LOO restitution


In [ ]:
for name in ("t_loo_results", "v_loo_metrics", "v_loo_method_comparison", "t_loo_hotels"):
    print("===", name, "===")
    if not cp.relation_exists(name):
        print("  (absente)")
        continue
    d = cp.table_view(name).df()
    display(d.head(40))
    print(" shape", d.shape)


## 9. `p_table_view` sur une vue restitution


In [ ]:
for name in (
    "v_restitution_default_input_mix",
    "v_restitution_solution_coefficients",
    "t_sales",
):
    print("===", name, "===")
    try:
        d = cp.p_table_view(name).df()
        display(d.head(15))
        print(" shape", d.shape)
    except Exception as exc:
        print(" ", exc)


## 10. Explorer une relation libre


In [ ]:
NAME = "t_dataset_pivot"
df = cp.table_view(NAME).df() if cp.relation_exists(NAME) else None
if df is not None:
    display(df.iloc[:10, :15])
else:
    print("relation absente:", NAME)


## 11. Fermer


In [ ]:
cp.close()
print("connexion fermee")
